In [65]:
from dotenv import load_dotenv
from openai import OpenAI


In [66]:
load_dotenv()
client = OpenAI()

In [67]:
question = input("사용자 요청 메시지")
response = client.chat.completions.create(
  model='gpt-4.1-mini',
  messages=[
    # AI 가 할 일
    {"role":"system","content":"You are a helpful assistant. using locale language."},
    # 사용자 요청
    {"role":"user","content":question}
  ],
)
print(f'🤖AI: {response.choices[0].message.content}')


🤖AI: Xin chào! Bạn cần trợ giúp gì hôm nay? Bạn có thể cho tôi biết rõ hơn về yêu cầu của bạn được không?


In [68]:
# 실시간 데이터는 LLM 의 학습 내용이 아닙니다.
# ㄴ 함수 또는 tools 사용해서 답변을 할 수 있는 기능이 있습니다.

# 시간 구하기 함수
from datetime import datetime

def get_current_time():
  now = datetime.now().strftime('%H:%M:%S')
  print(f'✅log : {now}')
  return now

get_current_time()

# 날짜 구하기 함수 : '%Y:%m:%d'
def get_current_date():
  now = datetime.now().strftime('%Y-%m-%d')
  print(f'✅log : {now}')
  return now

# gpt 가 사용할 지정된 함수 목록을 정의
myfunctions = [
  {
    "name": "get_current_time",
    "description": "현재 시간 출력합니다. 포맷 HH:MM:SS 입니다.",

    "name": "get_current_date",
    "description": "현재 날짜 출력합니다. 포맷 YYYY-mm-dd 입니다."
  }
]

✅log : 13:53:18


In [ ]:
question = '오늘 날짜를 알려줘' #'현재 시간을 알려줘'
response = client.chat.completions.create(
  model='gpt-4.1-mini',
  messages=[
    {"role":"system","content":"You are a helpful assistant. using locale language."},
    {"role":"user","content":question}
  ],
  functions=myfunctions,
)
# response 는 json 문자열 ChatCompletions 타입 객체. 그 안에 속성은 객체 타입 or 문자열
print(f'🔁log response: {response.model_dump_json(indent=2)}')


🔁log response: {
  "id": "chatcmpl-CYmHli97FwIDJ7TnxRhnakagMTbwe",
  "choices": [
    {
      "finish_reason": "function_call",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": null,
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": {
          "arguments": "{}",
          "name": "get_current_date"
        },
        "tool_calls": null
      }
    }
  ],
  "created": 1762404797,
  "model": "gpt-4.1-mini-2025-04-14",
  "object": "chat.completion",
  "service_tier": "default",
  "system_fingerprint": "fp_4c2851f862",
  "usage": {
    "completion_tokens": 11,
    "prompt_tokens": 60,
    "total_tokens": 71,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audio_tokens": 0,
      "reasoning_tokens": 0,
      "rejected_prediction_tokens": 0
    },
    "prompt_tokens_details": {
      "audio_tokens": 0,
      "cached_tokens": 0
    }
  }
}


In [64]:
# 함수 호출을 하고 답변을 만드는 추가적인 요청이 필요합니다.
import json
# fn_name = response.choices[0].message.function_call
# ↪ 🔁 loge fn_name: FunctionCall(arguments='{}', name='get_current_time')
fn_name = getattr(response.choices[0].message.function_call, 'name', None)
print(f'🔁 loge fn_name: {fn_name}')
if fn_name:
  func_response = globals()[fn_name]() # 문자열로 된 함수를 호출하는 방법
  followup_response =  client.chat.completions.create(
    model='gpt-4.1-mini',
    messages=[
    {"role":"system","content":"You are a helpful assistant. using locale language."},
    {"role":"user","content":f'{fn_name} 함수를 실행한 결과 {func_response} 이용하여 최종 응답 만들어줘.'}
  ],
  functions=myfunctions
  )
  print(followup_response.model_dump_json(indent=2))
  # 함수 실행한 두 번째 요청 응답
  result = followup_response.choices[0].message.content
else:
  # 첫 번째 요청 응답
  result = response.choices[0].message.content

print(f'🤖AI : {result}')

🔁 loge fn_name: get_current_date
✅log : 2025-11-06
{
  "id": "chatcmpl-CYkZQRoasFTftNufad2Yb1ydI1Wxc",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "현재 날짜는 2025년 11월 6일입니다.\n\n초코케이크 만드는 법도 알려드릴게요!\n\n재료:\n- 밀가루 200g\n- 설탕 150g\n- 코코아 가루 50g\n- 베이킹 파우더 1작은술\n- 소금 약간\n- 달걀 2개\n- 우유 120ml\n- 식용유 80ml\n- 바닐라 엑스트랙 1작은술\n- 뜨거운 물 120ml\n\n만드는 방법:\n1. 오븐을 180도로 예열합니다.\n2. 큰 볼에 밀가루, 설탕, 코코아 가루, 베이킹 파우더, 소금을 체에 쳐서 넣습니다.\n3. 다른 볼에 달걀, 우유, 식용유, 바닐라 엑스트랙을 넣고 잘 섞습니다.\n4. 마른 재료에 젖은 재료를 넣고 잘 섞어 반죽을 만듭니다.\n5. 뜨거운 물을 넣고 부드럽게 섞습니다.\n6. 반죽을 케이크 틀에 붓고 30~35분간 구워줍니다.\n7. 식힌 후 케이크를 꺼내어 데코레이션 하시면 완성입니다.\n\n맛있는 초코케이크 만드세요!",
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": null
      }
    }
  ],
  "created": 1762398204,
  "model": "gpt-4.1-mini-2025-04-14",
  "object": "chat.completion",
  "service_tier": "default",
 